# Experiment 2.8 counterpart notebook: phantom-rank toy study with pre-orthogonalization noise

This notebook is the analysis counterpart to `run_experiment.py` in the same directory. It **does not reimplement the training loop**. Instead, it imports the script module, runs the same default experiment, and presents the returned structured results in a more academically serious and notebook-safe form.

## Scope and truthful framing

This is still a **single-seed deep-linear toy study**. The current implementation measures:

- final loss and loss trajectories
- mean **effective rank** (entropy-based surrogate, not hard rank) of the **raw per-layer gradients**
- mean effective rank of the **pre-Newton-Schulz noisy gradients actually fed into Muon**

This notebook does **not** claim to measure hard numerical rank, post-orthogonalization update span, or literal “full parameter-space restoration.” Any interpretation below is limited to the measured erank/loss quantities returned by the script.


In [ ]:

import importlib.util
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

try:
    import pandas as pd
except ImportError:
    pd = None

plt.style.use('default')

EXPERIMENT_RELATIVE_PATH = Path('experiments/Tier_1_Core_Mechanism_Tests/PHANTOM_RANK_noise_injection/run_experiment.py')
FALLBACK_REPO_ROOT = Path('/home/secemp9/Muon_as_RG_Gauge_Fixing')


def find_repo_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    if FALLBACK_REPO_ROOT not in candidates:
        candidates.append(FALLBACK_REPO_ROOT)
    for candidate in candidates:
        if (candidate / EXPERIMENT_RELATIVE_PATH).exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing run_experiment.py')


def load_experiment_module(script_path: Path):
    spec = importlib.util.spec_from_file_location(
        'phantom_rank_noise_injection_experiment', script_path
    )
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def summary_rows_to_dataframe(summary_rows):
    rows = []
    for row in summary_rows:
        rows.append({
            'noise_label': row['noise_label'],
            'noise_frac': row['noise_frac'],
            'noise_seed': row['noise_seed'],
            'final_loss': row['final_loss'],
            'initial_raw_grad_mean_erank': row['initial_raw_grad_mean_erank'],
            'time_avg_raw_grad_mean_erank': row['time_avg_raw_grad_mean_erank'],
            'final_raw_grad_mean_erank': row['final_raw_grad_mean_erank'],
            'time_avg_pre_ns_noisy_grad_mean_erank': row['time_avg_pre_ns_noisy_grad_mean_erank'],
            'first_raw_grad_erank_below_half_step': row['first_raw_grad_erank_below_half_step'],
            'starts_below_half': row['starts_below_half'],
        })
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


def display_table_fallback(rows):
    if pd is not None:
        display(rows)
        return
    header = [
        'noise_label', 'noise_frac', 'noise_seed', 'final_loss',
        'initial_raw_grad_mean_erank', 'time_avg_raw_grad_mean_erank',
        'final_raw_grad_mean_erank', 'time_avg_pre_ns_noisy_grad_mean_erank',
        'first_raw_grad_erank_below_half_step', 'starts_below_half',
    ]
    markdown = [
        '| ' + ' | '.join(header) + ' |',
        '| ' + ' | '.join(['---'] * len(header)) + ' |',
    ]
    for row in rows:
        markdown.append('| ' + ' | '.join(str(row.get(col, '')) for col in header) + ' |')
    display(Markdown(chr(10).join(markdown)))


NOTEBOOK_START = time.perf_counter()
REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT_PATH = REPO_ROOT / EXPERIMENT_RELATIVE_PATH
experiment = load_experiment_module(SCRIPT_PATH)

print(f'Loaded experiment module from: {SCRIPT_PATH}')
print(f'Python: {sys.version.split()[0]}')
print(f'NumPy: {np.__version__}')
print(f'Matplotlib: {plt.matplotlib.__version__}')
pandas_msg = pd.__version__ if pd is not None else 'not installed (markdown-table fallback active)'
print(f'pandas: {pandas_msg}')


## Reproducibility, runtime, and configuration

The script now returns structured results, including the exact configuration and the deterministic per-noise RNG seeds used for the pre-orthogonalization noise. The old process-dependent `hash(str(noise_frac))` seeding has been removed, so repeated runs in separate Python processes should now agree exactly for the default configuration.


In [ ]:
run_start = time.perf_counter()
results = experiment.run_experiment()
notebook_run_elapsed = time.perf_counter() - run_start

config = results['config']
summary_rows = results['summary_rows']
summary_table = summary_rows_to_dataframe(summary_rows)
verdict = results['verdict']
per_noise = results['per_noise']
data_diagnostics = results['data_diagnostics']
noise_seed_map = {row['noise_label']: row['noise_seed'] for row in summary_rows}

print('Experiment identity: script-backed notebook counterpart for run_experiment.py')
print(f'Repo root: {REPO_ROOT}')
print(f'Script path: {SCRIPT_PATH}')
print(f'Script-reported runtime: {results["runtime_sec"]:.2f} s')
print(f'Notebook wall-clock run time for experiment call: {notebook_run_elapsed:.2f} s')
print()
print('Configuration:')
print(json.dumps({
    'width': config['width'],
    'depth': config['depth'],
    'num_steps': config['num_steps'],
    'lr_muon': config['lr_muon'],
    'ns_iters': config['ns_iters'],
    'batch_size': config['batch_size'],
    'seed': config['seed'],
    'track_every': config['track_every'],
    'noise_levels': config['noise_levels'],
    'noise_labels': config['noise_labels'],
    'half_rank_threshold': config['half_rank_threshold'],
}, indent=2))
print()
print('Deterministic noise seeds by condition:')
print(json.dumps(noise_seed_map, indent=2))
print()
print('Data diagnostics:')
print(json.dumps(data_diagnostics, indent=2)[:2000])


## Summary table of measured quantities

The table below is intentionally narrow: it reports only quantities the script actually computes.

- `initial_raw_grad_mean_erank` is the mean effective rank of the raw per-layer gradients at tracked step 0.
- `time_avg_raw_grad_mean_erank` is the time-average of that raw-gradient erank history.
- `time_avg_pre_ns_noisy_grad_mean_erank` is the time-average of the pre-Newton-Schulz noisy gradient erank actually fed into Muon during tracked updates.
- The `n/2` threshold is a **heuristic** for this toy study, not a theorem.


In [ ]:
if pd is not None:
    display(summary_table.round({
        'noise_frac': 4,
        'final_loss': 6,
        'initial_raw_grad_mean_erank': 2,
        'time_avg_raw_grad_mean_erank': 2,
        'final_raw_grad_mean_erank': 2,
        'time_avg_pre_ns_noisy_grad_mean_erank': 2,
    }))
else:
    display_table_fallback(summary_table)


In [ ]:
half_threshold = verdict['heuristic_half_rank_threshold']
baseline_initial = verdict['baseline_initial_raw_grad_mean_erank']
if verdict['baseline_initial_below_half']:
    baseline_message = (
        f'**Baseline threshold check:** the 0% baseline starts at raw-gradient mean erank '
        f'{baseline_initial:.2f}, already below the heuristic threshold n/2 = {half_threshold:.1f}. '
        'So this run is better described as an **already low-erank regime** than as a demonstrated collapse from near-full rank.'
    )
else:
    first_below = verdict['baseline_first_below_half_step']
    baseline_message = (
        f'**Baseline threshold check:** the 0% baseline starts above n/2 = {half_threshold:.1f} '
        f'and first drops below it at tracked step {first_below}.'
    )
display(Markdown(baseline_message))


## Figure 1 — Raw-gradient mean effective rank vs. training step

This figure shows the script’s tracked **raw-gradient** mean erank across layers for each noise condition. This is the most direct continuity check against the original toy study, but it is still only an entropy-based surrogate for dimensionality.


In [ ]:
plt.figure(figsize=(9, 5))
for noise_label in config['noise_labels']:
    run = per_noise[noise_label]
    plt.plot(
        run['raw_grad_mean_erank_steps'],
        run['raw_grad_mean_erank_history'],
        marker='o',
        markersize=3,
        linewidth=1.5,
        label=noise_label,
    )
plt.axhline(half_threshold, color='black', linestyle='--', linewidth=1.0, label='n/2 heuristic')
plt.xlabel('training step')
plt.ylabel('mean effective rank of raw layer gradients')
plt.title('Raw-gradient mean erank trajectories (entropy-based surrogate)')
plt.legend(title='noise level', ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
baseline_avg = per_noise['0%']['time_avg_raw_grad_mean_erank']
one_pct_avg = per_noise['1%']['time_avg_raw_grad_mean_erank']
display(Markdown(
    'The baseline raw-gradient erank is already below the heuristic threshold at step 0 in this default run, '
    f'with time-averaged raw-gradient mean erank {baseline_avg:.2f}. The 1% condition has time-averaged raw-gradient '
    f'mean erank {one_pct_avg:.2f}. This supports only a **modest shift in the measured surrogate**, not a claim of restored full-rank updates.'
))


## Figure 2 — Final loss vs. noise level

The original pair-level goal includes comparing final training loss across the noise sweep. This figure presents the actual measured endpoint loss for each run under the shared initialization and deterministic seeds.


In [ ]:
noise_labels = config['noise_labels']
final_losses = [per_noise[label]['final_loss'] for label in noise_labels]
x = np.arange(len(noise_labels))

plt.figure(figsize=(8, 4.8))
plt.plot(x, final_losses, marker='o', linewidth=1.8)
plt.xticks(x, noise_labels)
plt.xlabel('noise level (fraction of ||G||_F)')
plt.ylabel('final loss')
plt.title('Final loss across pre-orthogonalization noise levels')
for xi, loss in zip(x, final_losses):
    plt.annotate(f'{loss:.6f}', (xi, loss), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
best_noise = verdict['best_noise_label']
best_loss = verdict['best_final_loss']
one_pct_delta = verdict['one_percent_improvement_abs']
one_pct_pct = verdict['one_percent_improvement_pct']
display(Markdown(
    f'The best final loss in this deterministic single-seed sweep is **{best_noise}** with loss **{best_loss:.6f}**. '
    f'Relative to the 0% baseline, the 1% condition changes final loss by {one_pct_delta:.6f} ({one_pct_pct:.2f}%). '
    'This is the main performance signal currently available in the toy setup.'
))


## Figure 3 — Mean effective rank of the gradients actually fed into Newton–Schulz

A major reporting gap in the earlier version was that the notebook discussed noise lifting rank, but only tracked the **raw** gradients. The script now also exposes the mean effective rank of the **pre-NS noisy gradients** actually passed to Newton–Schulz during tracked updates.


In [ ]:
plt.figure(figsize=(9, 5))
for noise_label in config['noise_labels']:
    run = per_noise[noise_label]
    plt.plot(
        run['pre_ns_noisy_grad_mean_erank_steps'],
        run['pre_ns_noisy_grad_mean_erank_history'],
        marker='o',
        markersize=3,
        linewidth=1.5,
        label=noise_label,
    )
plt.axhline(half_threshold, color='black', linestyle='--', linewidth=1.0, label='n/2 heuristic')
plt.xlabel('training step')
plt.ylabel('mean effective rank of pre-NS noisy gradients')
plt.title('Pre-Newton–Schulz noisy-gradient mean erank trajectories')
plt.legend(title='noise level', ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
baseline_pre_ns = per_noise['0%']['time_avg_pre_ns_noisy_grad_mean_erank']
one_pct_pre_ns = per_noise['1%']['time_avg_pre_ns_noisy_grad_mean_erank']
display(Markdown(
    f'The 0% baseline has time-averaged pre-NS noisy-gradient mean erank {baseline_pre_ns:.2f} (identical to raw, because no noise is added). '
    f'The 1% condition has time-averaged pre-NS noisy-gradient mean erank {one_pct_pre_ns:.2f}. '
    'This directly measures the intervention point discussed in the narrative, but it still remains an erank surrogate rather than a post-orthogonalization span metric.'
))


## Conclusion tied to the measured outputs

The conclusion below is intentionally conservative and references only quantities computed by the script.


In [ ]:

conclusion_lines = [
    '### Script-backed conclusion',
    '',
    f'- Baseline 0% initial raw-gradient mean erank: **{verdict["baseline_initial_raw_grad_mean_erank"]:.2f}** versus heuristic threshold **n/2 = {verdict["heuristic_half_rank_threshold"]:.1f}**.',
    '- In this default run, the baseline is best described as **already in a low-erank regime at step 0**.' if verdict['baseline_is_already_low_erank_at_step0'] else '- In this default run, the baseline only enters the low-erank regime later in training.',
    f'- Best final loss in the tested sweep: **{verdict["best_noise_label"]}** with loss **{verdict["best_final_loss"]:.6f}**.',
    f'- 1% noise improves final loss relative to baseline: **{verdict["one_percent_improves_final_loss"]}**; delta = **{verdict["one_percent_improvement_abs"]:.6f}** ({verdict["one_percent_improvement_pct"]:.2f}%).',
    f'- 1% noise raises time-averaged raw-gradient mean erank: **{verdict["one_percent_raises_time_avg_raw_grad_mean_erank"]}**.',
    f'- 1% noise raises time-averaged pre-NS noisy-gradient mean erank: **{verdict["one_percent_raises_time_avg_pre_ns_noisy_grad_mean_erank"]}**.',
    '',
    '**Conservative reading:** ' + verdict['interpretation'],
    '',
    '**Caveat:** this does not establish hard-rank recovery, post-NS update coverage, or full parameter-space restoration; it is a single-seed toy result based on an entropy-style effective-rank surrogate.',
]
display(Markdown(chr(10).join(conclusion_lines)))
print(f'Total notebook elapsed time so far: {time.perf_counter() - NOTEBOOK_START:.2f} s')
